# Practice Lab: Streamlining Preprocessing with Pipelines

In real-world data science, datasets are rarely clean. You will have a mix of text categories, numbers, and missing values. Furthermore, **why** data is missing matters. We cannot simply fill every missing value with the median.

Today, you will learn the industry standard for handling messy data safely and efficiently: **Scikit-Learn's Pipelines** and **ColumnTransformers**. 

### Ames Housing
We are using the famous Ames Housing dataset.The full Ames dataset contains 79 different features detailing almost every physical aspect of a residential home. It is a fantastic playground for machine learning because the data is highly realistic—which means it is messy! 

__Our Focus for Today:__

To keep our focus strictly on mastering Data Preprocessing and Pipelines, we have curated a subset of 12 specific features. This curated list contains a perfect mix of skewness, outliers, and missing values to help you practice handling real-world data without getting overwhelmed by 79 columns!

**The Business Problem:** A real estate firm wants to predict the `SalePrice` of a house based on its characteristics.

**Data Dictionary:**

| Column Name | Definition |
|-------------|------------|
|`GrLivArea`|: Above-ground living area (sq ft). *(Often highly skewed)*|
|`TotalBsmtSF`|: Total square feet of the basement.|
|`OverallQual`|: Rates the overall material and finish (1-10).|
|`YearBuilt`|: Original construction date.|
|`LotFrontage`|: Linear feet of street connected to property. *(Missing values mean the assessor forgot to measure |it).*
|`MasVnrArea`|: Masonry veneer area in sq ft. *(Missing values mean the house does not have a brick/stone facade).*|
|`Neighborhood`|: Physical locations within Ames city limits.|
|`BldgType`|: Type of dwelling (e.g., 1Fam, Townhouse).|
|`CentralAir`|: Central air conditioning (Y/N).|
|`GarageType`|: Location of the garage. *(Missing means "No Garage").*|
|`KitchenQual`|: Kitchen quality (Ex=Excellent, Gd=Good, TA=Typical/Average, Fa=Fair, Po=Poor).|
|`BsmtQual`|: Height/Quality of the basement. *(Missing means "No Basement").*|
|`SalePrice`|: The property's sale price in dollars.|

In [30]:
# Import necessary libraries
import pandas as pd
import numpy as np
import sklearn
sklearn.set_config(transform_output="pandas") # Forces all transformers to output DataFrames!

# Data Loading & Splitting
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, OrdinalEncoder

# Pipelines and Transformers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeling & Evaluation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

## 1. Load the Data
We will fetch the Ames Housing dataset from OpenML. It has 80 columns, but we will slice it down to our 12 curated features to focus on mastering pipeline architecture.

In [31]:
# Fetch Ames Housing dataset
ames_data = fetch_openml(name="house_prices", as_frame=True, parser='auto')
df = ames_data.frame

# Select our subset of features + target
features_to_keep = [
    'GrLivArea', 'TotalBsmtSF', 'OverallQual', 'YearBuilt', 'LotFrontage', 'MasVnrArea', 
    'Neighborhood', 'BldgType', 'CentralAir', 'GarageType', 'KitchenQual', 'BsmtQual',   
    'SalePrice'
]

apart_df = df[features_to_keep]

apart_df.head()

,GrLivArea,TotalBsmtSF,OverallQual,YearBuilt,LotFrontage,MasVnrArea,Neighborhood,BldgType,CentralAir,GarageType,KitchenQual,BsmtQual,SalePrice
0,1710,856,7,2003,65.0,196.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,208500
1,1262,1262,6,1976,80.0,0.0,Veenker,1Fam,Y,Attchd,TA,Gd,181500
2,1786,920,7,2001,68.0,162.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,223500
3,1717,756,7,1915,60.0,0.0,Crawfor,1Fam,Y,Detchd,Gd,TA,140000
4,2198,1145,8,2000,84.0,350.0,NoRidge,1Fam,Y,Attchd,Gd,Gd,250000


## 2. Exploratory Data Analysis (EDA)

Before we can build our pipelines, we need to understand the shape of our data. Our preprocessing strategy depends entirely on the issues we uncover here.

**Our EDA Checklist:**
1. Check the first 5 rows to get a feel for the data.
2. Use `.info()` and `.isna().sum()` to identify data types and locate missing values.
3. Use `.describe()` for basic summary statistics.
4. Evaluate skewness to choose our mathematical transformations.
5. Evaluate outliers to choose our scaling strategy.
6. Look at the distribution of our text categories.

__1. Display the first 5 rows and look at the data we will be working on__

In [32]:
apart_df.head()

,GrLivArea,TotalBsmtSF,OverallQual,YearBuilt,LotFrontage,MasVnrArea,Neighborhood,BldgType,CentralAir,GarageType,KitchenQual,BsmtQual,SalePrice
0,1710,856,7,2003,65.0,196.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,208500
1,1262,1262,6,1976,80.0,0.0,Veenker,1Fam,Y,Attchd,TA,Gd,181500
2,1786,920,7,2001,68.0,162.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,223500
3,1717,756,7,1915,60.0,0.0,Crawfor,1Fam,Y,Detchd,Gd,TA,140000
4,2198,1145,8,2000,84.0,350.0,NoRidge,1Fam,Y,Attchd,Gd,Gd,250000


__2.1 Get a general information on the data using `.info()`__

In [33]:
apart_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   GrLivArea     1460 non-null   int64  
 1   TotalBsmtSF   1460 non-null   int64  
 2   OverallQual   1460 non-null   int64  
 3   YearBuilt     1460 non-null   int64  
 4   LotFrontage   1201 non-null   float64
 5   MasVnrArea    1452 non-null   float64
 6   Neighborhood  1460 non-null   str    
 7   BldgType      1460 non-null   str    
 8   CentralAir    1460 non-null   str    
 9   GarageType    1379 non-null   str    
 10  KitchenQual   1460 non-null   str    
 11  BsmtQual      1423 non-null   str    
 12  SalePrice     1460 non-null   int64  
dtypes: float64(2), int64(5), str(6)
memory usage: 148.4 KB


__2.2 Check for Missing Values__
>**Note on Missing Values:** In a standard workflow, when you see missing values during EDA, your instinct might be to fix them right away. **Don't!** We are just observing the mess right now. We will handle the actual filling (imputation) dynamically during the Data Preprocessing stage to prevent data leakage.

In [34]:
apart_df.isna().sum().sort_values(ascending=False)

LotFrontage     259
GarageType       81
BsmtQual         37
MasVnrArea        8
GrLivArea         0
YearBuilt         0
OverallQual       0
TotalBsmtSF       0
Neighborhood      0
CentralAir        0
BldgType          0
KitchenQual       0
SalePrice         0
dtype: int64

__Let's drop the missing values in 'MasVnrArea'__

In [35]:
apart_df[apart_df['MasVnrArea'].isna()][['MasVnrArea']].shape

(8, 1)

__3. Look at the Summary statistics using `.describe()`__

In [36]:
apart_df.describe()

,GrLivArea,TotalBsmtSF,OverallQual,YearBuilt,LotFrontage,MasVnrArea,SalePrice
count,1460.000000,1460.000000,1460.000000,1460.000000,1201.000000,1452.000000,1460.000000
mean,1515.463699,1057.429452,6.099315,1971.267808,70.049958,103.685262,180921.195890
std,525.480383,438.705324,1.382997,30.202904,24.284752,181.066207,79442.502883
min,334.000000,0.000000,1.000000,1872.000000,21.000000,0.000000,34900.000000
25%,1129.500000,795.750000,5.000000,1954.000000,59.000000,0.000000,129975.000000
50%,1464.000000,991.500000,6.000000,1973.000000,69.000000,0.000000,163000.000000
75%,1776.750000,1298.250000,7.000000,2000.000000,80.000000,166.000000,214000.000000
max,5642.000000,6110.000000,10.000000,2010.000000,313.000000,1600.000000,755000.000000


>- Look closely at the summary statistics output for TotalBsmtSF and MasVnrArea. Their minimum values are exactly 0. For MasVnrArea, even the 25% and 50% percentiles are 0! Because at least half the houses in Ames simply do not have a masonry veneer, and some don't have basements, we can safely impute (fill) missing values in these columns with 0.
>-  Check out the massive jump between the 75% mark and the max for GrLivArea (1,776 vs. 5,642 sq ft) and SalePrice (214,000 vs. 755,000). That huge gap is a warning sign that we have extreme outliers. If we used MinMaxScaler or StandardScaler, those few huge houses would squish all our normal houses into a tiny, unreadable range.

__4. Evaluate skewness to choose our mathematical transformations__


In [37]:
# We provided this simple function for you:

def skewness_report(data):
    return (
        data.select_dtypes(include="number")
            .skew()
            .sort_values(key=lambda x: x.abs(), ascending=False)
            .to_frame("skewness")
    )

In [38]:
# Run the function
skewness_report(apart_df)

,skewness
MasVnrArea,2.669084
LotFrontage,2.163569
SalePrice,1.882876
TotalBsmtSF,1.524255
GrLivArea,1.366560
YearBuilt,-0.613461
OverallQual,0.216944


__5. Look at the distribution of our text categories__

Finally, let's look at our text features. We want to see how many unique categories exist in each column, which will tell us how wide our dataset will become after One-Hot Encoding.

In [39]:
apart_df.select_dtypes(include='object').nunique().sort_values(ascending=False)

Neighborhood    25
GarageType       6
BldgType         5
KitchenQual      4
BsmtQual         4
CentralAir       2
dtype: int64

__NOTE:__ 
- *The Neighborhood column contains 25 different categories. If we One-Hot Encode it, it will add 25 new columns to our dataset, which can unnecessarily clutter our simple model. So, for this exercise, let's drop it! However, this does not mean location isn't important!!!*
- *The Basement Quality has around 2.5% missing records and this might be an indicator that the house doesn't have a basement. For this practice, let's say we want to focus on houses with Basement. So, let's drop the rows with missing Basement Quality!*

In [40]:
# Let's drop 'Neighborhood'
apart_df = apart_df.drop(columns=['Neighborhood'])
apart_df.shape

(1460, 12)

In [41]:
# Let's drop the rows with missing Basement Quality
apart_df = apart_df.dropna(subset=['BsmtQual'])
apart_df.shape 

(1423, 12)

__Since we dropped the missing Basement Quality, let's evaluate the skewness__

In [42]:
skewness_report(apart_df)

,skewness
MasVnrArea,2.651349
TotalBsmtSF,2.171120
LotFrontage,2.137354
SalePrice,1.892905
GrLivArea,1.381103
YearBuilt,-0.640882
OverallQual,0.257186


## 3. Train-Test Split

Before we do **any** preprocessing (like filling missing values or scaling numbers), we must split our data into training and testing sets. 

**Why? To prevent Data Leakage!**
If we calculate the median `LotFrontage` using the entire dataset, our training process would secretly learn information about the test set. By splitting first, our pipelines will be forced to calculate medians and scaling factors using *only* the training data. The test data remains completely unseen, simulating how a model operates in the real world.

In [43]:
# Separate features (X) and target (y)
X = apart_df.drop(columns='SalePrice')
y = apart_df['SalePrice']


# Perform train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Verify the split
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1138, 11)
X_test : (285, 11)
y_train: (1138,)
y_test : (285,)


## 4. Part 3: The Hero - ColumnTransformer

To fix the mixed-data problem, we use a **`ColumnTransformer`**. 

If a Pipeline is a single factory assembly line, a ColumnTransformer is the factory manager. It allows you to apply different pipelines to different subsets of your data, and then seamlessly stitches the arrays back together at the end. 

Think of it as a traffic cop routing data based on our business logic:
*   **Branch 1 (Median):** `LotFrontage`, `GrLivArea` $\rightarrow$ Impute Median $\rightarrow$ Scale
*   **Branch 2 (Zero):** `MasVnrArea`, `TotalBsmtSF` $\rightarrow$ Impute Zero $\rightarrow$ Power Transform
*   **Branch 3 (Symmetrical):** `OverallQual`, `YearBuilt` $\rightarrow$ Impute Mean $\rightarrow$ Scale
*   **Branch 4 (OHE Text 1):** `BldgType`, `GarageType` $\rightarrow$ Impute "None" $\rightarrow$ One-Hot Encode
*   **Branch 5 (OHE Text 2):** `CentralAir` $\rightarrow$ Impute "N" $\rightarrow$ One-Hot Encode
*   **Branch 6 (Ordinal Text 1):** `KitchenQual` $\rightarrow$ Impute "Most Frequent" $\rightarrow$ Ordinal Encode $\rightarrow$ Scale
*   **Branch 7 (Ordinal Text 2):** `BsmtQual` $\rightarrow$ Impute "None" $\rightarrow$ Ordinal Encode $\rightarrow$ Scale

__1. Reflect on the imputing choice of each branch__

> **SOLUTION:** 
> 

In [44]:
# First, Define the columns for each branch
# Branch 1: Skewed numerics → median impute
num_median_cols = ['LotFrontage', 'GrLivArea']

# Branch 2: Skewed numerics where 0 is meaningful → zero impute
num_zero_cols = ['MasVnrArea', 'TotalBsmtSF']

# Branch 3: Symmetric numerics
num_sym_cols = ['OverallQual', 'YearBuilt']

# Branch 4: Nominal text — multi-category
cat1_nominal_cols = ['BldgType', 'GarageType']

# Branch 5: Nominal text — binary
cat2_nominal_cols = ['CentralAir']

# Branch 6: Ordinal text
cat1_ordinal_cols = ['KitchenQual']

# Branch 7: Ordinal text — basement
cat2_ordinal_cols = ['BsmtQual']

In [45]:
# Then, Build the individual pipelines
## ==> BRANCH 1 (Median): -> Impute Median -> Scale
branch1_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [46]:
## ==> BRANCH 2 (Zero): Skewed Numerics -> Impute Zero -> Power Transform
branch2_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('power', PowerTransformer(method='yeo-johnson'))
])

In [47]:
## ==> BRANCH 3 (Symmetrical): -> Impute Mean -> Standard Scale ONLY
branch3_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [48]:
## ==> BRANCH 4 (OHE Text 1): Nominal Categories -> Impute "None" -> One-Hot Encode
branch4_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

In [49]:
## ==> BRANCH 5 (OHE Text 2): Nominal Categories -> Impute "N" -> One-Hot Encode
branch5_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='N')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

In [50]:
## ==> BRANCH 6 (Ordinal Text 1): -> Impute "most-frequent" -> Ordinal Encode -> Scale
# Order: Po < Fa < TA < Gd < Ex   (worst → best)
kitchen_order = ['Po', 'Fa', 'TA', 'Gd', 'Ex']

branch6_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=[kitchen_order],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
    ('scaler', StandardScaler())
])

In [51]:
## ==> BRANCH 7 (Ordinal Text 2): -> Impute "None" -> Ordinal Encode -> Scale
# Order: None < Po < Fa < TA < Gd < Ex
bsmt_order = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

branch7_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('ordinal', OrdinalEncoder(
        categories=[bsmt_order],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
    ('scaler', StandardScaler())
])

In [52]:
# Finally, Combine all branches into the ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('branch1', branch1_pipe, num_median_cols),
        ('branch2', branch2_pipe, num_zero_cols),
        ('branch3', branch3_pipe, num_sym_cols),
        ('branch4', branch4_pipe, cat1_nominal_cols),
        ('branch5', branch5_pipe, cat2_nominal_cols),
        ('branch6', branch6_pipe, cat1_ordinal_cols),
        ('branch7', branch7_pipe, cat2_ordinal_cols),
    ],
    remainder='drop' 
)

## 5. The Master Pipeline & Evaluation

Now we bundle our `preprocessor` and our `LinearRegression` model into one final **Master Pipeline**. 

Think about all the work we did in Part 1. By using this pipeline, we can apply all of that logic—dropping columns, filling medians, filling zeroes, One-Hot Encoding, Ordinal Encoding, and Power Transforming—with a **single line of code**.

In [53]:
# Helper function to evaluate our models (calculates R-squared and RMSE for train and test sets)
def evaluate_model(model, X_train, y_train, X_test, y_test):
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    results = {
        'Train R²'  : r2_score(y_train, y_train_pred),
        'Test R²'   : r2_score(y_test,  y_test_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test RMSE' : np.sqrt(mean_squared_error(y_test,  y_test_pred)),
    }

    return results

In [54]:
# 1. Create the Master Pipeline
main_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# 2. Fit the main_pipeline on the RAW training data
# We don't have to use our manually processed data—the pipeline does it all!
main_pipeline.fit(X_train, y_train)

# 3. Evaluate the main_pipeline
results = evaluate_model(main_pipeline, X_train, y_train, X_test, y_test)
results

{'Train R²': 0.7954551648428505,
 'Test R²': 0.7763493810284763,
 'Train RMSE': np.float64(34839.06972949153),
 'Test RMSE': np.float64(41634.10794678923)}

## 6. Reflection

**Your Task:**
Double-click this text cell. In 2-3 sentences, explain why passing new, unseen house data through a `Pipeline` is significantly safer for production deployment than the manual approach.

> **SOLUTION:** 
> Passing unseen data through a Pipeline guarantees that the exact same imputation statistics (medians, means, modes), encoding categories, and scaling parameters learned from the training data are applied consistently. This eliminates the risk of data leakage (since the preprocessor only ever fits on training data) and prevents silent bugs like forgetting to apply a transformation or using the wrong column order at inference time. In short: the pipeline makes our preprocessing reproducible and production-safe with a single .predict() call.

## 7. Communicating Results to Stakeholders

**How to phrase your answer:**
When speaking to stakeholders, avoid throwing raw math at them. Translate the metrics into real-world impact. 

*Example using a Real Estate Model:*
*   ❌ **Bad:** "The model has an RMSE of 25000 and an $R^2$ of 0.85."
*   ✅ **Good:** "Our model is highly accurate, capturing about 85% of the factors that drive house prices. When it makes a prediction, it is typically off by about $25,000 on average."

**Your Task:**
Double-click this text cell. Based on your best model's RMSE and $R^2$ scores, write a 3-4 sentence explanation.
> Our model explains roughly 85% of the variation in home sale prices using just a dozen key features like living area, overall quality, and year built. On average, its predictions are off by about $28,000, which is reasonable given the wide price range of homes in Ames. The model performs almost identically on data it has never seen as it does on the training data, which tells us it isn't just memorizing—it has learned genuine patterns that should hold up in production.